# Backtest: the earlier models on the `case2s` dataset

**Why:** the `case2s` pair files (`Overture/03_case2_street_swap.ipynb`) replace a slice of easy negatives with look-alike-street negatives. Every earlier model was scored on the old Overture set, where real case2 has only 15 test rows. This notebook puts all of them side by side on the new set, so one table shows how each one does on the harder slice and whether anything else moved.

**What it does:** loads the new files, rebuilds the similarity features, **trains** the classical models (Step 1, Step 1b) on the new train split, runs the two libpostal rules (no training), and scores the saved Step 3 siamese model (**trained on the old train split**; see §6). Everything is scored on the new val and test splits.

**How to read the table:** each cell is the percentage of that slice answered correctly, as `test (val)`. Positive slices = % found, negative slices = % rejected. `overall (accuracy)` mixes both and is dominated by the easy rows, so don't quote it alone. Val was used to choose settings, so val is slightly optimistic; test is the number to report. The old-dataset rows are in `results.md`.

Run `Overture/03` → `04` first. The first run builds the features (~2 min) and the libpostal predictions (the slow part, cached in `libpostal_results/`).

## 0. Setup
**Why:** one place for imports and the dataset names every cell uses.

In [1]:
# Setup: imports and dataset names
%load_ext autoreload
%autoreload 2

import os
import sys
from importlib import import_module
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import pandas as pd
from sklearn.linear_model import LogisticRegression
from torch.utils.data import DataLoader

from helpers.pairs import load_pairs, add_number_street
from helpers.features import get_processed_dfs
from helpers.check_model_performance import review_typical_slices

# digit-prefixed filenames -> can't use `from ... import`
run_against_models = import_module("model_baseline.01_process_dumb_models").run_against_models
_get_libpostal_results = import_module("model_baseline.04_dumb_libpostal_model")._get_libpostal_results
_real_step = import_module("siamese_model.13_real_training")
load_run, rebuild_model, THRESHOLD = _real_step.load_run, _real_step.rebuild_model, _real_step.THRESHOLD
_data_step = import_module("siamese_model.11_data_plumbing")
PairDataset, make_collate = _data_step.PairDataset, _data_step.make_collate
predict = import_module("siamese_model.12_training_step").predict

pd.set_option("display.width", 250)
pd.set_option("display.expand_frame_repr", False)

DATASET = "overture_case2s"   # helpers/pairs.py name for Overture/Data/ct_pairs_{split}_case2s.parquet
LIBPOSTAL_NAME = "Case2s"     # part of the libpostal prediction-cache filenames, so it never mixes with Overture / Stamford

## 1. Load the `case2s` pairs
**Why:** all later steps read the same three frames, with the house-number / street columns the features need.  
**Does:** `load_pairs` per split, then `add_number_street`. The table shows rows per split and `neg_type`; each split should be 1 positive : 9 negatives.  
**Output:** `train`, `val`, `test` (+ `*_label`)

In [2]:
# Load: the case2s train / val / test pairs
train, train_label = load_pairs(DATASET, "train")
val, val_label = load_pairs(DATASET, "val")
test, test_label = load_pairs(DATASET, "test")
train, val, test = [add_number_street(df) for df in [train, val, test]]

pd.concat([train, val, test]).groupby(["split", "neg_type"], observed = True).size().unstack()

neg_type,case0,case1,case2,case2s,case3
split,,,,,
train,20259,67174,172,5441,109522
val,4187,13781,34,894,22969
test,4386,14448,15,1053,23942


## 2. Similarity features
**Why:** the classical models read a table of string-similarity numbers per pair (Jaro-Winkler on street, address and ZIP, plus house-number and other features), not the raw strings. Building them is slow, so they are cached.  
**Does:** `get_processed_dfs` builds the features once per split and stores them in `case2s_training/` (git-ignored). The cache stores a hash of the pairs and raises if the pair files change: delete the cache then.  
**Output:** `train_X`, `val_X`, `test_X`

In [3]:
# Features: cached similarity features for the three splits
os.makedirs("./case2s_training", exist_ok = True)
train_X, val_X, test_X = get_processed_dfs([
    [train, "./case2s_training/case2s_evaluation_train.parquet"],
    [val, "./case2s_training/case2s_evaluation_val.parquet"],
    [test, "./case2s_training/case2s_evaluation_test.parquet"],
])

## 3. Step 1 and Step 1b models (trained on the `case2s` train split)
**Why:** these are the baselines the later models are measured against. They are retrained here, so the numbers show what each one learns when the training set itself contains look-alike-street negatives.  
**Does:** Step 1 = logistic regression on the three Jaro-Winkler features only. Step 1b = logistic regression, random forest and XGBoost on all 23 features (default settings, as before). Each is fitted on `train_X` and predicts val and test.  
**Output:** `preds` = `{model name: (val predictions, test predictions)}`

In [4]:
# Step 1 (3 features) and Step 1b (LR / RF / XGB on 23 features): fit on train, predict val and test
STEP1_COLS = ["jw_street_sim", "jw_address_sim", "jw_zipcode_sim"]
step1 = LogisticRegression().fit(train_X[STEP1_COLS], train_label)
preds = {"LogReg (3 Jaro-Winkler features) [1]": (step1.predict(val_X[STEP1_COLS]), step1.predict(test_X[STEP1_COLS]))}

ret = run_against_models(train_X, train_label, val_X, test_X)   # {model: {"val": [preds, probs], "test": [preds, probs]}}
for model, name in [("LogisticRegression", "LogReg (23 similarity features) [1b]"),
                    ("RandomForestClassifier", "Random forest [1b]"),
                    ("XGBClassifier", "XGBoost [1b]")]:
    preds[name] = (ret[model]["val"][0], ret[model]["test"][0])

## 4. libpostal rules (no training)
**Why:** libpostal is a rule, not a trained model, so it has nothing to learn from the new train split. It still has to be re-run, because the new val and test contain different pairs. These are the two settings chosen on val in Step 2c: strict/strict/strict (the rejection end) and loose/off/loose (the recall end).  
**Does:** `_get_libpostal_results` runs the rule per row and caches the predictions in `libpostal_results/` under `Case2s`, so a second run is instant. **The first run is slow** (libpostal parses every address).  
**Output:** two more entries in `preds`

In [5]:
# libpostal: the two rule settings on val and test (cached)
for road, zip_rule, number in [("strict", "strict", "strict"), ("loose", "off", "loose")]:
    v = _get_libpostal_results(val, road, zip_rule, number, "val", dataset = LIBPOSTAL_NAME)["pred"]
    t = _get_libpostal_results(test, road, zip_rule, number, "test", dataset = LIBPOSTAL_NAME)["pred"]
    preds[f"libpostal, {road}/{zip_rule}/{number} [2c]"] = (v, t)

## 5. Step 3 siamese model (saved checkpoint)
**Why:** to see how the character-level siamese model does on the new slice. **This model was trained on the OLD train split** (`overture`, no swaps), so its `case2s` score shows how it handles look-alike streets it never saw as negatives. Retraining it on `case2s` takes up to ~80 min (≈ 4 min per epoch), so that is a separate run, not part of this notebook.  
**Does:** loads the saved run (mean pooling, seed 0, the one in `results.md`), rebuilds the model from the checkpoint's own vocabulary and config, predicts val and test with threshold 0.5.  
**Output:** one more entry in `preds`

In [9]:
# Siamese: saved checkpoint (trained on the old train split) scored on the new val and test
def siamese_preds(ckpt, df, labels):
    model = rebuild_model(ckpt)
    loader = DataLoader(PairDataset(df, labels), batch_size = 64, shuffle = False, collate_fn = make_collate(ckpt["vocab"]))
    return (predict(model, loader) >= THRESHOLD).int().numpy()

history, ckpt = load_run(0, "mean", checkpoint_dir = "../siamese_model/checkpoints")
preds["Siamese BiLSTM, trained on the OLD train [3]"] = (siamese_preds(ckpt, val, val_label), siamese_preds(ckpt, test, test_label))

## 5b. Step 3 siamese model retrained on `case2s`
**Why:** the row above is trained on the old train split, so it can't be compared fairly with the classical models (retrained on `case2s`). This row is the same model and protocol trained on the `case2s` train split.  
**Does:** loads the checkpoint written by `siamese_model/15_train_on_case2s.ipynb` (`checkpoints_case2s/`) and scores val and test the same way. If that notebook hasn't been run, it prints a message and the table simply has no such row.  
**Output:** one more entry in `preds`

In [10]:
# Siamese retrained on case2s (checkpoint from siamese_model/15_train_on_case2s.ipynb), if it exists
if os.path.exists("../siamese_model/checkpoints_case2s/overture_siamese_seed_0_pooling_mean.pt"):
    _, ckpt_case2s = load_run(0, "mean", checkpoint_dir = "../siamese_model/checkpoints_case2s")
    preds["Siamese BiLSTM, trained on case2s [3]"] = (siamese_preds(ckpt_case2s, val, val_label), siamese_preds(ckpt_case2s, test, test_label))
else:
    print("no case2s checkpoint yet: run siamese_model/15_train_on_case2s.ipynb first")

## 6. Results table
**Why:** one table with every model, in the same layout as `results.md`.  
**Does:** for each model, % correct per slice on test and val, written as `test (val)`; the last row is the number of test pairs in each slice. Float tables stay in `pct_tables` if you want to sort or subtract.  
**Output:** `table`

In [11]:
# Results: % correct per slice, "test (val)", one row per model
def pct(df, model_preds, labels):
    col = review_typical_slices(df, model_preds, labels).set_index("slice")["% correct"]
    return col.str[:-1].astype(float)

pct_tables = {name: (pct(val, v, val_label), pct(test, t, test_label)) for name, (v, t) in preds.items()}
table = pd.DataFrame({
    name: pt.map("{:.1f}".format) + " (" + pv.map("{:.1f}".format) + ")" for name, (pv, pt) in pct_tables.items()
}).T
first_test_preds = next(iter(preds.values()))[1]
table.loc["test n"] = review_typical_slices(test, first_test_preds, test_label).set_index("slice")["n"].map("{:,}".format)
table

slice,positives: identical,positives: non-identical,"neg: case1 (same street, diff number)","neg: case2 (same number, similar street)",neg: case3 (easy),neg: case2s (swapped look-alike street),overall (accuracy)
LogReg (3 Jaro-Winkler features) [1],99.9 (100.0),59.2 (62.4),99.7 (99.5),80.0 (61.8),100.0 (100.0),59.6 (63.4),98.1 (98.3)
LogReg (23 similarity features) [1b],99.9 (100.0),50.5 (57.1),99.1 (99.2),66.7 (47.1),100.0 (100.0),74.4 (77.0),98.1 (98.4)
Random forest [1b],100.0 (100.0),70.1 (76.8),99.6 (99.5),53.3 (55.9),99.9 (99.9),88.7 (91.3),99.0 (99.2)
XGBoost [1b],100.0 (100.0),69.9 (76.3),99.6 (99.5),53.3 (52.9),100.0 (100.0),88.5 (90.2),99.0 (99.1)
"libpostal, strict/strict/strict [2c]",99.1 (99.0),52.3 (56.5),100.0 (100.0),80.0 (88.2),100.0 (100.0),97.4 (98.2),98.9 (99.0)
"libpostal, loose/off/loose [2c]",100.0 (100.0),71.9 (75.7),97.9 (98.3),73.3 (50.0),98.9 (99.4),75.1 (74.9),97.6 (98.0)
"Siamese BiLSTM, trained on the OLD train [3]",100.0 (100.0),83.6 (87.1),99.4 (99.4),33.3 (29.4),99.9 (99.9),22.2 (24.8),97.6 (97.9)
"Siamese BiLSTM, trained on case2s [3]",100.0 (100.0),80.1 (83.1),99.3 (99.3),53.3 (64.7),100.0 (100.0),75.7 (74.5),98.8 (98.8)
test n,"3,543",843,"14,448",15,"23,942","1,053","43,844"


## Diagnostics
One-off inspections, kept out of the pipeline cells above. Results are written into the markdown (question → result → query).